# Notebook 10 — In Silico SAR Optimization

Starting from our two best T790M-relevant candidates, we systematically replace
aromatic C-H bonds with common medicinal chemistry substituents and measure
whether any modification improves T790M binding selectivity.

| Parent | Role | pharma | delta (baseline) |
|--------|------|--------|-----------------|
| **Rank #18** | Pharmacophore-confirmed, EGFR-like scaffold | 4/5 | -0.23 |
| **Rank #4**  | Strongest docking on T790M (-10.7) | 2/5 | -1.23 |
| **Rank #9**  | NH+AmiPy scaffold, moderate T790M preference | 3/5 | -0.40 |
| **Rank #11** | Same features as Rank #9, near-neutral start | 3/5 | -0.08 |
| **Rank #12** | Strongest absolute binder (-11.4) | 2/5 | -1.01 |

**Substituents screened:** F, Cl, CH3, OCH3, OH, NH2, CN, CF3

**Docking:** AutoDock Vina exh=8 on both 1M17 (WT) and 4I22 (T790M)

**Runtime:** CPU only. ~3-5 hours total (resumable on disconnect).


In [ ]:
# -- Bootstrap --
import os, sys, subprocess, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
except ImportError:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

# Install dependencies
for pkg in ['rdkit', 'pandas', 'matplotlib', 'tqdm']:
    try: __import__(pkg)
    except: subprocess.run([sys.executable,'-m','pip','install','-q',pkg])

if not shutil.which('vina'):
    subprocess.run(['apt-get','install','-y','-qq','autodock-vina','openbabel'])
if not shutil.which('obabel'):
    subprocess.run(['apt-get','install','-y','-qq','openbabel'])

import rdkit, pandas as pd, numpy as np, matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, QED as QEDmod, rdMolDescriptors

print('PROJECT_ROOT :', PROJECT_ROOT)
print('rdkit        :', rdkit.__version__)
print('vina         :', shutil.which('vina') or 'MISSING')
print('obabel       :', shutil.which('obabel') or 'MISSING')


## Step 1 — Load parent molecules and set up docking

In [ ]:
# -- Load parent molecules and build Vina dockers --
import json, yaml
from src.docking import VinaDocker

# Load top-20 and get Rank #18 and Rank #4
top20 = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv')

rank18 = top20[top20.index == 17].iloc[0]   # 0-indexed row 17 = rank 18
rank4  = top20[top20.index == 3].iloc[0]    # 0-indexed row 3  = rank 4

PARENTS = {
    'rank18': {
        'rank': 18,
        'smiles': rank18['smiles'],
        'vina_WT': rank18['vina_WT'],
        'vina_Mut': rank18['vina_Mut'],
        'delta': rank18['delta'],
        'QED': rank18['QED'],
        'pharma': 4,
        'label': 'Rank18 (pharma=4/5)',
    },
    'rank4': {
        'rank': 4,
        'smiles': rank4['smiles'],
        'vina_WT': rank4['vina_WT'],
        'vina_Mut': rank4['vina_Mut'],
        'delta': rank4['delta'],
        'QED': rank4['QED'],
        'pharma': 2,
        'label': 'Rank4 (pharma=2/5)',
    },
}

for key, p in PARENTS.items():
    mol = Chem.MolFromSmiles(p['smiles'])
    print(f"{p['label']}")
    print(f"  SMILES   : {p['smiles'][:70]}...")
    print(f"  vina_Mut = {p['vina_Mut']:.2f}  delta = {p['delta']:+.3f}  QED = {p['QED']:.2f}")
    print()

# Vina dockers
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)

docker_wt = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
    center=tuple(centers['1M17']),
    box_size=(20.0, 20.0, 20.0),
    exhaustiveness=8,
)
docker_mut = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/4I22_receptor.pdbqt',
    center=tuple(centers['4I22']),
    box_size=(20.0, 20.0, 20.0),
    exhaustiveness=8,
)
print('Dockers ready.')
print('WT  receptor:', docker_wt.receptor)
print('Mut receptor:', docker_mut.receptor)


## Step 2 — Generate analogs by aromatic substitution

For each parent molecule, we find all aromatic C-H positions and replace each
with 8 common medicinal chemistry substituents. Invalid molecules (failed
sanitization) and molecules violating MW > 600 are discarded.


In [ ]:
# -- Analog generation via aromatic C-H substitution --
from rdkit import Chem
from rdkit.Chem import Descriptors

SUBSTITUENTS = {
    'F':    'F',
    'Cl':   'Cl',
    'CH3':  'C',
    'OCH3': 'OC',
    'OH':   'O',
    'NH2':  'N',
    'CN':   'C#N',
    'CF3':  'C(F)(F)F',
}

MAX_POSITIONS = 5   # limit to first N aromatic CH positions per molecule

def get_aromatic_ch_positions(mol):
    patt = Chem.MolFromSmarts('[c;H1]')
    return [m[0] for m in mol.GetSubstructMatches(patt)]

def make_analog(mol, pos_idx, sub_smi):
    sub_mol = Chem.MolFromSmiles(sub_smi)
    if sub_mol is None:
        return None
    try:
        combined = Chem.CombineMols(mol, sub_mol)
        em = Chem.EditableMol(combined)
        em.AddBond(pos_idx, mol.GetNumAtoms(), Chem.BondType.SINGLE)
        result = em.GetMol()
        Chem.SanitizeMol(result)
        # Roundtrip to canonical SMILES to validate
        smi = Chem.MolToSmiles(result)
        back = Chem.MolFromSmiles(smi)
        if back is None:
            return None
        return smi
    except Exception:
        return None

def generate_analogs(parent_key, parent_smi, substituents, max_pos=MAX_POSITIONS):
    mol = Chem.MolFromSmiles(parent_smi)
    if mol is None:
        return []

    positions = get_aromatic_ch_positions(mol)[:max_pos]
    print(f'  {parent_key}: {len(get_aromatic_ch_positions(mol))} aromatic CH positions found, '
          f'using first {len(positions)}')

    seen = {Chem.MolToSmiles(mol)}
    records = []
    for pos in positions:
        for sub_name, sub_smi in substituents.items():
            new_smi = make_analog(mol, pos, sub_smi)
            if new_smi is None or new_smi in seen:
                continue
            # Drug-likeness check: MW <= 650
            new_mol = Chem.MolFromSmiles(new_smi)
            if new_mol is None:
                continue
            mw = Descriptors.ExactMolWt(new_mol)
            if mw > 650:
                continue
            seen.add(new_smi)
            records.append({
                'parent_key':  parent_key,
                'parent_smi':  parent_smi,
                'pos_idx':     pos,
                'substituent': sub_name,
                'smiles':      new_smi,
                'MW':          round(mw, 1),
            })
    return records

# Generate analogs for both parents
all_analogs = []
for key, p in PARENTS.items():
    print(f"Generating analogs for {p['label']}:")
    analogs = generate_analogs(key, p['smiles'], SUBSTITUENTS)
    all_analogs.extend(analogs)
    print(f"  -> {len(analogs)} valid analogs generated")

analogs_df = pd.DataFrame(all_analogs)
print(f"\nTotal analogs: {len(analogs_df)}")
print(analogs_df['parent_key'].value_counts())
print(analogs_df['substituent'].value_counts())


## Step 3 — Cross-dock all analogs (resumable)

Each analog is docked against both WT and T790M pockets. Checkpoints every 10
molecules. Re-running this cell after a disconnect will skip completed analogs.


In [ ]:
# -- Cross-dock all analogs (resumable) --
from src.docking import cross_dock_smiles_list
import os

out_dir = f'{PROJECT_ROOT}/results/analysis/sar_docking'
os.makedirs(out_dir, exist_ok=True)

# Dock each parent group separately (easier to resume per parent)
sar_results = {}
for parent_key in ['rank18', 'rank4']:
    sub_df = analogs_df[analogs_df['parent_key'] == parent_key]
    csv_path = f'{out_dir}/{parent_key}_analogs.csv'

    print(f"\nDocking {len(sub_df)} analogs for {parent_key}...")
    results = cross_dock_smiles_list(
        smiles_list=sub_df['smiles'].tolist(),
        docker_wt=docker_wt,
        docker_mut=docker_mut,
        output_csv=csv_path,
        source_label=f'SAR_{parent_key}',
        checkpoint_every=10,
    )
    # Merge substituent info back
    results = results.merge(
        sub_df[['smiles','substituent','pos_idx','MW']],
        on='smiles', how='left'
    )
    # Add parent baseline for comparison
    parent_delta = PARENTS[parent_key]['delta']
    results['delta_vs_parent'] = results['delta'] - parent_delta

    results.to_csv(csv_path, index=False)
    sar_results[parent_key] = results
    print(f"  Saved {len(results)} docked analogs -> {csv_path}")


## Step 4 — Analyze and visualize SAR results

In [ ]:
# -- SAR analysis: delta comparison and best variants --
from rdkit.Chem import Draw
from IPython.display import display
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
best_variants = {}

for idx, (parent_key, results) in enumerate(sar_results.items()):
    ax = axes[idx]
    p = PARENTS[parent_key]
    ok = results[(results['status_WT']=='ok') & (results['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    if ok.empty:
        print(f'{parent_key}: no successful dockings yet')
        continue

    sub_means = ok.groupby('substituent')['delta'].agg(['mean','std','count']).reset_index()
    sub_means = sub_means.sort_values('mean')
    colors = ['#27ae60' if m < p['delta'] else '#e74c3c' for m in sub_means['mean']]

    ax.barh(sub_means['substituent'], sub_means['mean'], color=colors, alpha=0.75)
    ax.axvline(p['delta'], color='black', linestyle='--', linewidth=1.5,
               label='Parent delta = ' + f'{p["delta"]:+.2f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_xlabel('Mean Selectivity Delta (kcal/mol)')
    ax.set_title('SAR - ' + p["label"] + ' (green = improves T790M selectivity)')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

    best_row = ok.loc[ok['delta'].idxmin()]
    best_variants[parent_key] = best_row
    print('\n=== ' + p['label'] + ' ===')
    print('  Parent baseline     : delta = ' + f'{p["delta"]:+.3f}')
    print('  Best analog delta   : ' + f'{best_row["delta"]:+.3f}' + '  (' + str(best_row['substituent']) + ' sub)')
    print('  Improvement         : ' + f'{best_row["delta"] - p["delta"]:+.3f}' + ' kcal/mol')
    print('  Best analog SMILES  : ' + best_row['smiles'][:70] + '...')
    print()
    print('  Substituent breakdown (mean delta):')
    print(sub_means[['substituent','mean','count']].to_string(index=False))

plt.tight_layout()
out_fig = f'{PROJECT_ROOT}/results/analysis/sar_substituent_comparison.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out_fig)


In [ ]:
# -- Structure grid: parent vs best analogs --
from rdkit.Chem import Draw
from IPython.display import display
import pandas as pd

for parent_key, results in sar_results.items():
    p = PARENTS[parent_key]
    ok = results[(results['status_WT']=='ok') & (results['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    if ok.empty:
        continue

    # Top-5 most T790M-selective analogs
    top5 = ok.nsmallest(5, 'delta')

    mols = [Chem.MolFromSmiles(p['smiles'])]  # parent first
    parent_leg = ('PARENT rank #' + str(p['rank']) +
                  ' delta=' + f'{p["delta"]:+.2f}' +
                  ' vina_Mut=' + f'{p["vina_Mut"]:.1f}')
    legs = [parent_leg]

    for _, row in top5.iterrows():
        mol = Chem.MolFromSmiles(row['smiles'])
        mols.append(mol)
        leg = ('Sub: ' + str(row['substituent']) +
               ' delta=' + f'{row["delta"]:+.2f}' +
               ' (parent: ' + f'{p["delta"]:+.2f}' + ')' +
               ' vina_Mut=' + f'{row["vina_Mut"]:.1f}')
        legs.append(leg)

    valid_pairs = [(m, l) for m, l in zip(mols, legs) if m is not None]
    mols_v, legs_v = zip(*valid_pairs)

    print('\n--- ' + p['label'] + ': Parent + Top-5 T790M-selective analogs ---')
    grid = Draw.MolsToGridImage(list(mols_v), molsPerRow=3,
                                 subImgSize=(300, 230), legends=list(legs_v))
    display(grid)
    out_path = f'{PROJECT_ROOT}/results/analysis/sar_grid_{parent_key}.png'
    try:
        grid.save(out_path)
    except AttributeError:
        with open(out_path, 'wb') as f:
            f.write(grid.data)
    print('Saved ->', out_path)


## Step 5 — Summary

In [ ]:
# -- SAR summary for report --
import pandas as pd

print("=" * 65)
print("SAR OPTIMIZATION SUMMARY")
print("=" * 65)

for parent_key, results in sar_results.items():
    p = PARENTS[parent_key]
    ok = results[(results['status_WT']=='ok') & (results['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']

    if ok.empty:
        print(f'{p["label"]}: no results yet'); continue

    n_improved = (ok['delta'] < p['delta']).sum()
    best = ok.loc[ok['delta'].idxmin()]

    print(f"\n{p['label']}")
    print(f"  Parent delta    : {p['delta']:+.3f} kcal/mol")
    print(f"  Analogs docked  : {len(ok)}")
    print(f"  Analogs improved: {n_improved}/{len(ok)} "
          f"({100*n_improved/len(ok):.0f}%)")
    print(f"  Best analog     : delta={best['delta']:+.3f} "
          f"via {best['substituent']} substitution")
    print(f"  Max improvement : {best['delta'] - p['delta']:+.3f} kcal/mol")

    # Best by substituent type
    print("  By substituent (mean delta):")
    for _, row in ok.groupby('substituent')['delta'].mean().sort_values().items():
        marker = '<<' if row < p['delta'] else '  '
        print(f"    {_:8s}: {row:+.3f} {marker}")

print()
print("All results saved to results/analysis/sar_docking/")
print("Figures saved to results/analysis/sar_*.png")


## Optional: Include best analogs in the selectivity scatter

If any analog significantly outperforms its parent, add it to `combined_scores.csv`
so it appears on the final selectivity scatter in notebook 09.


In [ ]:
# -- Optionally merge best analogs into combined_scores.csv --
import os, pandas as pd

DELTA_THRESHOLD = -0.8   # include analogs with delta < this threshold

all_good = []
for parent_key, results in sar_results.items():
    p = PARENTS[parent_key]
    ok = results[(results['status_WT']=='ok') & (results['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    ok['source'] = f'SAR_{parent_key}'
    better = ok[ok['delta'] < DELTA_THRESHOLD]
    if not better.empty:
        all_good.append(better[['source','smiles','vina_WT','vina_Mut','delta',
                                  'status_WT','status_Mut']])
        print(f"{parent_key}: {len(better)} analogs pass delta < {DELTA_THRESHOLD}")

if all_good:
    new_rows = pd.concat(all_good, ignore_index=True)
    combined_path = f'{PROJECT_ROOT}/results/vina_scores/combined_scores.csv'
    existing = pd.read_csv(combined_path)
    updated = pd.concat([existing, new_rows], ignore_index=True)
    updated.to_csv(combined_path, index=False)
    print(f"Added {len(new_rows)} SAR analogs to combined_scores.csv")
else:
    print(f"No analogs surpassed delta threshold of {DELTA_THRESHOLD}.")
    print("This is itself a meaningful result: single-point substitutions")
    print("do not significantly improve T790M selectivity, consistent with")
    print("our main finding that selectivity requires deeper scaffold changes.")


---
## SAR Extension: Rank #9 (pharma=3/5)

In [ ]:
# -- SAR Extension: Rank #9 (pharma=3/5, delta=-0.395) --
import os, pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from IPython.display import display
import matplotlib.pyplot as plt

top20 = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv')
rank9_row = top20.iloc[8]  # 0-indexed row 8 = rank #9

PARENTS['rank9'] = {
    'rank': 9, 'smiles': rank9_row['smiles'],
    'vina_WT': rank9_row['vina_WT'], 'vina_Mut': rank9_row['vina_Mut'],
    'delta': rank9_row['delta'], 'QED': rank9_row['QED'],
    'pharma': 3, 'label': 'Rank9 (pharma=3/5)',
}
p9 = PARENTS['rank9']
print(f"Rank #9  delta={p9['delta']:+.3f}  vina_Mut={p9['vina_Mut']:.2f}  QED={p9['QED']:.2f}")

print("\nGenerating analogs for rank9...")
rank9_analogs = generate_analogs('rank9', p9['smiles'], SUBSTITUENTS)
rank9_df = pd.DataFrame(rank9_analogs)
print(f"  -> {len(rank9_analogs)} valid analogs")

out_dir = f'{PROJECT_ROOT}/results/analysis/sar_docking'
csv_path = f'{out_dir}/rank9_analogs.csv'
from src.docking import cross_dock_smiles_list
print(f"Docking {len(rank9_analogs)} analogs for rank9...")
results9 = cross_dock_smiles_list(
    smiles_list=rank9_df['smiles'].tolist(),
    docker_wt=docker_wt, docker_mut=docker_mut,
    output_csv=csv_path, source_label='SAR_rank9', checkpoint_every=10,
)
results9['delta'] = results9['vina_Mut'] - results9['vina_WT']
results9['delta_vs_parent'] = results9['delta'] - p9['delta']
if 'substituent' not in results9.columns:
    results9 = results9.merge(rank9_df[['smiles','substituent','pos_idx','MW']], on='smiles', how='left')
results9.to_csv(csv_path, index=False)
sar_results['rank9'] = results9

ok9 = results9[(results9['status_WT']=='ok') & (results9['status_Mut']=='ok')]
n_better = (ok9['delta'] < p9['delta']).sum()
print(f"rank9:  {len(ok9)} docked | delta [{ok9['delta'].min():.2f}, {ok9['delta'].max():.2f}]")
print(f"  improved: {n_better}/{len(ok9)}")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 5))
sub_means = ok9.groupby('substituent')['delta'].mean().reset_index().sort_values('delta')
colors = ['#27ae60' if m < p9['delta'] else '#e74c3c' for m in sub_means['delta']]
ax.barh(sub_means['substituent'], sub_means['delta'], color=colors, alpha=0.75)
ax.axvline(p9['delta'], color='black', linestyle='--', linewidth=1.5,
           label='Parent delta = ' + f'{p9["delta"]:+.2f}')
ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Mean Selectivity Delta (kcal/mol)')
ax.set_title('SAR - Rank9 (pharma=3/5)  (green = improves T790M selectivity)')
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
out_bar = f'{PROJECT_ROOT}/results/analysis/sar_rank9_bar.png'
plt.savefig(out_bar, dpi=150, bbox_inches='tight')
plt.show()

# Structure grid
top5_9 = ok9.nsmallest(5, 'delta')
mols = [Chem.MolFromSmiles(p9['smiles'])]
legs = ['PARENT rank #9  delta=' + f'{p9["delta"]:+.2f}' + '  vina_Mut=' + f'{p9["vina_Mut"]:.1f}']
for _, row in top5_9.iterrows():
    mols.append(Chem.MolFromSmiles(row['smiles']))
    legs.append('Sub: ' + str(row['substituent']) + '  delta=' + f'{row["delta"]:+.2f}' +
                '  vina_Mut=' + f'{row["vina_Mut"]:.1f}')
valid = [(m,l) for m,l in zip(mols,legs) if m is not None]
mv, lv = zip(*valid)
grid = Draw.MolsToGridImage(list(mv), molsPerRow=3, subImgSize=(300,230), legends=list(lv))
display(grid)
try: grid.save(f'{PROJECT_ROOT}/results/analysis/sar_grid_rank9.png')
except AttributeError:
    with open(f'{PROJECT_ROOT}/results/analysis/sar_grid_rank9.png','wb') as f: f.write(grid.data)


---
## SAR Extension: Rank #11 + Rank #12

In [ ]:
# -- SAR Extension: Rank #11 (pharma=3/5) + Rank #12 (pharma=2/5) --
import os, pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from IPython.display import display
import matplotlib.pyplot as plt
from src.docking import cross_dock_smiles_list

top20 = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv')
PARENTS['rank11'] = {
    'rank': 11, 'smiles': top20.iloc[10]['smiles'],
    'vina_WT': top20.iloc[10]['vina_WT'], 'vina_Mut': top20.iloc[10]['vina_Mut'],
    'delta': top20.iloc[10]['delta'], 'QED': top20.iloc[10]['QED'],
    'pharma': 3, 'label': 'Rank11 (pharma=3/5)',
}
PARENTS['rank12'] = {
    'rank': 12, 'smiles': top20.iloc[11]['smiles'],
    'vina_WT': top20.iloc[11]['vina_WT'], 'vina_Mut': top20.iloc[11]['vina_Mut'],
    'delta': top20.iloc[11]['delta'], 'QED': top20.iloc[11]['QED'],
    'pharma': 2, 'label': 'Rank12 (pharma=2/5)',
}
for key in ['rank11','rank12']:
    p = PARENTS[key]
    print(f"{p['label']}:  delta={p['delta']:+.3f}  vina_Mut={p['vina_Mut']:.2f}  QED={p['QED']:.2f}")

out_dir = f'{PROJECT_ROOT}/results/analysis/sar_docking'
os.makedirs(out_dir, exist_ok=True)

for key in ['rank11','rank12']:
    p = PARENTS[key]
    csv_path = f'{out_dir}/{key}_analogs.csv'
    analogs = generate_analogs(key, p['smiles'], SUBSTITUENTS)
    if not analogs:
        print(f"  {key}: no substitutable positions found"); continue
    df_a = pd.DataFrame(analogs)
    print(f"\nDocking {len(analogs)} analogs for {key}...")
    results = cross_dock_smiles_list(
        smiles_list=df_a['smiles'].tolist(),
        docker_wt=docker_wt, docker_mut=docker_mut,
        output_csv=csv_path, source_label=f'SAR_{key}', checkpoint_every=10,
    )
    results['delta'] = results['vina_Mut'] - results['vina_WT']
    results['delta_vs_parent'] = results['delta'] - p['delta']
    if 'substituent' not in results.columns:
        results = results.merge(df_a[['smiles','substituent','pos_idx','MW']], on='smiles', how='left')
    results.to_csv(csv_path, index=False)
    sar_results[key] = results
    ok = results[(results['status_WT']=='ok') & (results['status_Mut']=='ok')]
    n_imp = (ok['delta'] < p['delta']).sum()
    print(f"  {len(ok)} docked | delta [{ok['delta'].min():.2f}, {ok['delta'].max():.2f}]  improved: {n_imp}/{len(ok)}")

# Bar charts
done_keys = [k for k in ['rank11','rank12'] if k in sar_results]
if done_keys:
    fig, axes = plt.subplots(1, len(done_keys), figsize=(7*len(done_keys), 5))
    if len(done_keys) == 1: axes = [axes]
    for ax, key in zip(axes, done_keys):
        p = PARENTS[key]
        ok = sar_results[key]
        ok = ok[(ok['status_WT']=='ok') & (ok['status_Mut']=='ok')].copy()
        ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
        sub_means = ok.groupby('substituent')['delta'].mean().reset_index().sort_values('delta')
        colors = ['#27ae60' if m < p['delta'] else '#e74c3c' for m in sub_means['delta']]
        ax.barh(sub_means['substituent'], sub_means['delta'], color=colors, alpha=0.75)
        ax.axvline(p['delta'], color='black', linestyle='--', linewidth=1.5,
                   label='Parent delta = ' + f'{p["delta"]:+.2f}')
        ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
        ax.set_xlabel('Mean Selectivity Delta (kcal/mol)')
        ax.set_title('SAR - ' + p['label'])
        ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_ROOT}/results/analysis/sar_rank11_rank12_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

# Structure grids
for key in done_keys:
    p = PARENTS[key]
    ok = sar_results[key]
    ok = ok[(ok['status_WT']=='ok') & (ok['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    top5 = ok.nsmallest(5, 'delta')
    mols = [Chem.MolFromSmiles(p['smiles'])]
    legs = ['PARENT rank #' + str(p['rank']) + '  delta=' + f'{p["delta"]:+.2f}' + '  vina_Mut=' + f'{p["vina_Mut"]:.1f}']
    for _, row in top5.iterrows():
        mols.append(Chem.MolFromSmiles(row['smiles']))
        legs.append('Sub: ' + str(row.get('substituent','?')) + '  delta=' + f'{row["delta"]:+.2f}' + '  vina_Mut=' + f'{row["vina_Mut"]:.1f}')
    valid = [(m,l) for m,l in zip(mols,legs) if m is not None]
    mv, lv = zip(*valid)
    grid = Draw.MolsToGridImage(list(mv), molsPerRow=3, subImgSize=(300,230), legends=list(lv))
    display(grid)
    out_g = f'{PROJECT_ROOT}/results/analysis/sar_grid_{key}.png'
    try: grid.save(out_g)
    except AttributeError:
        with open(out_g,'wb') as f: f.write(grid.data)


---
## Finalize: Save All SAR Outputs

In [ ]:
# -- Final Save: verify all SAR outputs + 5-way summary --
import os, pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

out_dir = f'{PROJECT_ROOT}/results/analysis/sar_docking'
ana_dir = f'{PROJECT_ROOT}/results/analysis'

# Reload all CSVs
sar_results_loaded = {}
for key in ['rank18','rank4','rank9','rank11','rank12']:
    csv_path = f'{out_dir}/{key}_analogs.csv'
    if os.path.exists(csv_path):
        res = pd.read_csv(csv_path)
        if 'delta' not in res.columns:
            res['delta'] = res['vina_Mut'] - res['vina_WT']
        sar_results_loaded[key] = res

# Save 5-way summary CSV
rows = []
for key in ['rank18','rank4','rank9','rank11','rank12']:
    if key not in sar_results_loaded: continue
    p   = PARENTS[key]
    ok  = sar_results_loaded[key]
    ok  = ok[(ok['status_WT']=='ok') & (ok['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    pb  = p['delta']
    n   = len(ok)
    imp = (ok['delta'] < pb).sum()
    best_row = ok.loc[ok['delta'].idxmin()]
    rows.append({
        'rank': p['rank'], 'label': p['label'], 'pharma_score': p['pharma'],
        'parent_delta': pb, 'parent_vina_Mut': p['vina_Mut'], 'parent_QED': p['QED'],
        'n_analogs': n, 'n_improved': imp,
        'improvement_rate': round(imp/n, 3) if n else 0,
        'best_delta': round(best_row['delta'], 3),
        'best_substituent': best_row.get('substituent','?'),
        'best_vina_Mut': round(best_row['vina_Mut'], 2),
        'delta_improvement': round(best_row['delta'] - pb, 3),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(f'{ana_dir}/sar_summary.csv', index=False)

# Regenerate missing structure grids
for key in ['rank18','rank4','rank9','rank11','rank12']:
    grid_path = f'{ana_dir}/sar_grid_{key}.png'
    if os.path.exists(grid_path): continue
    if key not in sar_results_loaded: continue
    p  = PARENTS[key]
    ok = sar_results_loaded[key]
    ok = ok[(ok['status_WT']=='ok') & (ok['status_Mut']=='ok')].copy()
    ok['delta'] = ok['vina_Mut'] - ok['vina_WT']
    top5 = ok.nsmallest(5, 'delta')
    mols = [Chem.MolFromSmiles(p['smiles'])]
    legs = ['PARENT rank #' + str(p['rank']) + '  delta=' + f'{p["delta"]:+.2f}']
    for _, row in top5.iterrows():
        mols.append(Chem.MolFromSmiles(row['smiles']))
        legs.append('Sub: ' + str(row.get('substituent','?')) + '  delta=' + f'{row["delta"]:+.2f}')
    valid = [(m,l) for m,l in zip(mols,legs) if m is not None]
    mv, lv = zip(*valid)
    grid = Draw.MolsToGridImage(list(mv), molsPerRow=3, subImgSize=(280,210), legends=list(lv))
    try: grid.save(grid_path)
    except AttributeError:
        with open(grid_path,'wb') as f: f.write(grid.data)

# Checklist
print('='*55); print('FINAL CHECKLIST'); print('='*55)
expected = (
    [f'{out_dir}/{k}_analogs.csv' for k in ['rank18','rank4','rank9','rank11','rank12']] +
    [f'{ana_dir}/sar_grid_{k}.png' for k in ['rank18','rank4','rank9','rank11','rank12']] +
    [f'{ana_dir}/sar_summary.csv',
     f'{ana_dir}/sar_substituent_comparison.png',
     f'{ana_dir}/sar_rank9_bar.png',
     f'{ana_dir}/sar_rank11_rank12_bar.png']
)
for f in expected:
    name = os.path.basename(f)
    status = '[OK]     ' if os.path.exists(f) else '[MISSING]'
    sz = f'  ({os.path.getsize(f)/1024:.1f} KB)' if os.path.exists(f) else ''
    print(f'  {status} {name}{sz}')

print()
print('5-WAY SAR SUMMARY:')
print(summary_df[['label','parent_delta','best_delta','n_improved','n_analogs','best_substituent']].to_string(index=False))
